# 🦴 KBG Hackathon 2025 — Bone Fracture Classifier (Colab GPU)

**Steps:**
1. Upload `KBG_Upload.zip` to your Google Drive root
2. Change runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
3. Run all cells top-to-bottom
4. Download results from `results/` when done

In [1]:
# Cell 1: Check GPU
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: User cancelled dfs_ephemeral authorization

In [ ]:
# Cell 3: Unzip the project (adjust path if you put it in a subfolder)
import os

ZIP_PATH = '/content/drive/MyDrive/KBG_Upload.zip'  # <-- Change if needed

assert os.path.exists(ZIP_PATH), f"Zip not found at {ZIP_PATH}! Upload it to Google Drive first."

!unzip -q -o "$ZIP_PATH" -d /content/kbg
print("✅ Unzipped successfully")
!ls /content/kbg/

In [ ]:
# Cell 4: Install dependencies
!pip install -q timm albumentations pyyaml scikit-learn matplotlib seaborn tqdm opencv-python-headless

In [ ]:
# Cell 5: Patch config.yaml for Colab paths + enable Colab-optimal settings
import yaml

PROJECT_DIR = '/content/kbg/KBG_Hackathon_Submission/bone_fracture_classifier'
DATA_DIR = '/content/kbg/data/clean_split'
CONFIG_PATH = os.path.join(PROJECT_DIR, 'config.yaml')

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Fix data paths for Colab
cfg['data']['data_dir']  = DATA_DIR
cfg['data']['train_dir'] = os.path.join(DATA_DIR, 'train')
cfg['data']['val_dir']   = os.path.join(DATA_DIR, 'val')
cfg['data']['test_dir']  = os.path.join(DATA_DIR, 'test')

# Colab-optimal settings (CUDA GPU available)
cfg['data']['num_workers'] = 2           # Colab has 2 CPU cores
cfg['data']['pin_memory']  = True
cfg['training']['batch_size'] = 32        # T4 has 16 GB — plenty for this
cfg['training']['accumulation_steps'] = 2  # effective batch = 64
cfg['training']['mixed_precision'] = True  # fp16 works on CUDA
cfg['training']['epochs'] = 25

# Use bigger models since T4 has 16 GB VRAM
cfg['model']['architecture'] = 'vit_small_patch16_224'
cfg['model']['ensemble']['models'] = [
    'vit_small_patch16_224',
    'efficientnet_b3',
    'convnext_small'
]
cfg['model']['ensemble']['weights'] = [0.5, 0.25, 0.25]

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print("✅ Config patched for Colab")
print(f"   Models: {cfg['model']['ensemble']['models']}")
print(f"   Batch: {cfg['training']['batch_size']} × {cfg['training']['accumulation_steps']} = {cfg['training']['batch_size'] * cfg['training']['accumulation_steps']}")
print(f"   Epochs: {cfg['training']['epochs']}")
print(f"   Data: {DATA_DIR}")

In [ ]:
# Cell 6: Verify data is there
for split in ['train', 'val', 'test']:
    for cls in ['fractured', 'not fractured']:
        p = os.path.join(DATA_DIR, split, cls)
        n = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f"{split:6s} / {cls:15s}: {n:5d} images")

In [ ]:
# Cell 7: Train + Evaluate + Auto-save to Drive 🚀
import os, shutil

PROJECT_DIR = '/content/kbg/KBG_Hackathon_Submission/bone_fracture_classifier'
DRIVE_OUTPUT = '/content/drive/MyDrive/KBG_Results'

# --- TRAIN ---
%cd $PROJECT_DIR
!python train.py --config config.yaml

# --- EVALUATE ---
!python evaluate.py --config config.yaml

# --- AUTO-SAVE TO DRIVE (immediately after training) ---
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

for folder in ['results', 'checkpoints', 'logs']:
    src = os.path.join(PROJECT_DIR, folder)
    dst = os.path.join(DRIVE_OUTPUT, folder)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✅ Saved {folder}/ to Drive")

shutil.copy2(os.path.join(PROJECT_DIR, 'config.yaml'), os.path.join(DRIVE_OUTPUT, 'config.yaml'))

print(f"\n🎉 ALL RESULTS SAVED TO: {DRIVE_OUTPUT}")
!ls -la "$DRIVE_OUTPUT"/results/
!ls -la "$DRIVE_OUTPUT"/checkpoints/

In [ ]:
# Cell 8: (Re-evaluate only — if you need to re-run evaluation without retraining)
import os, shutil
PROJECT_DIR = '/content/kbg/KBG_Hackathon_Submission/bone_fracture_classifier'
DRIVE_OUTPUT = '/content/drive/MyDrive/KBG_Results'

%cd $PROJECT_DIR
!python evaluate.py --config config.yaml

# Save updated results to Drive
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
src = os.path.join(PROJECT_DIR, 'results')
if os.path.exists(src):
    shutil.copytree(src, os.path.join(DRIVE_OUTPUT, 'results'), dirs_exist_ok=True)
    print("✅ Results saved to Drive")

In [ ]:
# Cell 9: Run 5-fold Cross-Validation
%cd /content/kbg/KBG_Hackathon_Submission/bone_fracture_classifier
!python train.py --config config.yaml --cv

In [ ]:
# Cell 10: Show results
import os
import pandas as pd
from IPython.display import display, Image

PROJECT_DIR = '/content/kbg/KBG_Hackathon_Submission/bone_fracture_classifier'

# Show metrics
results_csv = os.path.join(PROJECT_DIR, 'results', 'final_results.csv')
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    display(df)

# Show confusion matrix
cm_path = os.path.join(PROJECT_DIR, 'results', 'confusion_matrix.png')
if os.path.exists(cm_path):
    display(Image(cm_path))

# Show ROC curves
roc_path = os.path.join(PROJECT_DIR, 'results', 'roc_curves.png')
if os.path.exists(roc_path):
    display(Image(roc_path))

# Show GradCAM
gc_path = os.path.join(PROJECT_DIR, 'results', 'gradcam_samples.png')
if os.path.exists(gc_path):
    display(Image(gc_path))

In [ ]:
# Cell 11: Copy results back to Google Drive for download
import os, shutil

PROJECT_DIR = '/content/kbg/KBG_Hackathon_Submission/bone_fracture_classifier'
CONFIG_PATH = os.path.join(PROJECT_DIR, 'config.yaml')
DRIVE_OUTPUT = '/content/drive/MyDrive/KBG_Results'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy results folder
src_results = os.path.join(PROJECT_DIR, 'results')
if os.path.exists(src_results):
    shutil.copytree(src_results, os.path.join(DRIVE_OUTPUT, 'results'), dirs_exist_ok=True)

# Copy best checkpoint
src_ckpt = os.path.join(PROJECT_DIR, 'checkpoints')
if os.path.exists(src_ckpt):
    shutil.copytree(src_ckpt, os.path.join(DRIVE_OUTPUT, 'checkpoints'), dirs_exist_ok=True)

# Copy config used
shutil.copy2(CONFIG_PATH, os.path.join(DRIVE_OUTPUT, 'config.yaml'))

print(f"✅ All results saved to Google Drive: {DRIVE_OUTPUT}")
!ls -la "$DRIVE_OUTPUT"/results/

In [ ]:
# Cell 12: (Alternative) Download results directly to your laptop
from google.colab import files

# Zip the results
!cd /content/kbg/KBG_Hackathon_Submission/bone_fracture_classifier && zip -r /content/KBG_Results.zip results/ checkpoints/ config.yaml
files.download('/content/KBG_Results.zip')